### Train oracle model

In [1]:

from models import mlp_oracle
import torch
from torch.utils.data import DataLoader
import numpy as np
import os
import mavenn

from torch.utils.data import random_split

2026-02-17 09:41:33.845281: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset = mavenn.load_example_dataset('gb1')
x = dataset['x']
y = dataset['y']


In [3]:
# One hot encode the sequences
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
vocab_size = len(AMINO_ACIDS)  # 20 amino acids
aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def sequences_to_indices(sequences):
    """Convert string sequences to 2D array of indices."""
    seq_len = len(sequences[0])
    indices = np.zeros((len(sequences), seq_len), dtype=np.int32)
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            indices[i, j] = aa_to_idx.get(aa, 0)
    return indices

def one_hot_encode(sequences, vocab_size):
    """One-hot encode sequences (2D array of indices)."""
    one_hot = np.zeros((sequences.shape[0], sequences.shape[1], vocab_size), dtype=np.float32)
    for i in range(sequences.shape[0]):
        for j in range(sequences.shape[1]):
            aa_index = sequences[i, j]
            if aa_index < vocab_size:
                one_hot[i, j, aa_index] = 1.0
    return one_hot

# Convert string sequences to indices, then one-hot encode
x_indices = sequences_to_indices(x)
x_one_hot = one_hot_encode(x_indices, vocab_size)
print(f"Shape: {x_one_hot.shape}")  # Should be (n_samples, seq_len, 20)

Shape: (530737, 55, 20)


In [4]:
# Flatten one-hot encoded data: (n_samples, seq_len, vocab_size) -> (n_samples, seq_len * vocab_size)
x_flat = x_one_hot.reshape(x_one_hot.shape[0], -1)  # Shape: (n_samples, 80)
y_float = y.astype(np.float32)  # Convert to float32 for PyTorch
print(f"Flattened shape: {x_flat.shape}")


full_dataset = list(zip(x_flat, y_float))
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
print(f"Train size: {train_size}, Val size: {val_size}")

Flattened shape: (530737, 1100)
Train size: 424589, Val size: 106148


In [5]:
def train_oracle_model(train_loader, val_loader, model_save_path):
    """Train the MLP oracle model on the GB1 dataset with validation and save the trained model."""

    # Initialize model - reduced hidden size for regularization
    input_size = x_flat.shape[1]
    output_size = 1
    print(f"Input size: {input_size}")
    model = mlp_oracle.MLPOracle(input_size, output_size, hidden_size=256, dropout_rate=0.3)
    
    # Fit normalization parameters (on training data only)
    print("Fitting normalization parameters...")
    model.fit_normalization(train_loader)
    
    # Train the model with validation and early stopping
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train_model(train_loader, val_loader=val_loader, device=device, 
                      epochs=100, weight_decay=1e-4, patience=15)
    
    # Save the trained model parameters
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    model.save_model(model_save_path)

train_oracle_model(train_loader, val_loader, model_save_path="models/oracle_mlp.pth")

Input size: 1100
Fitting normalization parameters...
Epoch 1/100, Train Loss: 1.0450, Val Loss: 0.5932
Epoch 2/100, Train Loss: 0.6271, Val Loss: 0.4662
Epoch 3/100, Train Loss: 0.5691, Val Loss: 0.4680
Epoch 4/100, Train Loss: 0.5430, Val Loss: 0.5186
Epoch 5/100, Train Loss: 0.5271, Val Loss: 0.4588
Epoch 6/100, Train Loss: 0.5173, Val Loss: 0.5458
Epoch 7/100, Train Loss: 0.5104, Val Loss: 0.5011
Epoch 8/100, Train Loss: 0.5035, Val Loss: 0.4864
Epoch 9/100, Train Loss: 0.4983, Val Loss: 0.4804
Epoch 10/100, Train Loss: 0.4940, Val Loss: 0.4479
Epoch 11/100, Train Loss: 0.4877, Val Loss: 0.5597
Epoch 12/100, Train Loss: 0.4844, Val Loss: 0.5460
Epoch 13/100, Train Loss: 0.4797, Val Loss: 0.4594
Epoch 14/100, Train Loss: 0.4774, Val Loss: 0.4862
Epoch 15/100, Train Loss: 0.4738, Val Loss: 0.4867
Epoch 16/100, Train Loss: 0.4710, Val Loss: 0.6090
Epoch 17/100, Train Loss: 0.4689, Val Loss: 0.5679
Epoch 18/100, Train Loss: 0.4670, Val Loss: 0.5468
Epoch 19/100, Train Loss: 0.4635, Val 

In [6]:
# test inference
print(x_flat.shape[1])
oracle = mlp_oracle.MLPOracle(input_size=x_flat.shape[1], output_size=1, hidden_size=256, dropout_rate=0.3)
oracle.load_model('models/oracle_mlp.pth')
sequence = torch.tensor(x_flat[0], dtype=torch.float32).unsqueeze(0)  # Shape: (1, 80)
prediction = oracle.inference(sequence)
print(f"Predicted binding score: {prediction.item():.4f}, True binding score: {y[0]:.4f}")

1100
Predicted binding score: -2.8986, True binding score: -3.1452


In [9]:
# test inference on a mutated sequence
sequence = list(x[0])  # Original sequence as a list of characters
sequence[0] = 'V'
sequence[1] = 'V'
sequence[2] = 'V'  # Mutate the first amino acid to 'V
mutated_sequence = ''.join(sequence)  # Convert back to string
print(f"Original sequence: {x[0]}, Mutated sequence: {mutated_sequence}")
# Convert mutated sequence to one-hot encoding
mutated_indices = sequences_to_indices([mutated_sequence])  # Shape: (1, seq_len)
mutated_one_hot = one_hot_encode(mutated_indices, vocab_size)  #
mutated_flat = mutated_one_hot.reshape(mutated_one_hot.shape[0], -1)  # Shape: (1, 80)
mutated_tensor = torch.tensor(mutated_flat, dtype=torch.float32)  # Shape:
mutated_prediction = oracle.inference(mutated_tensor)
print(f"Predicted binding score for mutated sequence: {mutated_prediction.item():.4f}")

Original sequence: AAKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE, Mutated sequence: VVVLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE
Predicted binding score for mutated sequence: -1.6533
